In [ ]:
#!pip install transformers datasets accelerate evaluate torch tqdm


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm


In [ ]:
dataset = load_dataset("csv", data_files={"train": "debug_data.csv"})
dataset = dataset["train"]

def preprocess_function(example):
    input_text = f"Fix the following SysML code.\n### Faulty Code:\n{example['wrong_code']}\n### Error:\n{example['error']}"
    target_text = example["correct_code"]
    return {"input_text": input_text, "target_text": target_text}

dataset = dataset.map(preprocess_function)

In [ ]:
model_name = "Salesforce/codet5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


In [ ]:
def tokenize_function(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target_text"],
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(tokenize_function, batched=False)


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=data_collator
)


In [ ]:
batch = next(iter(train_dataloader)) # sanity check
for k, v in batch.items():
    print(k, v.shape)
